# A New Horizontal Integer Unpacking Algorithm

## Overview

When integers are stored in files like Apache Parquet, they are compressed by stripping their unused high-order bits (a technique called integer packing). Reading them back (unpacking) is a hot path in data pipelines, since large datasets are queried far more often than they are written. This post dives into a new unpacking algorithm that is fully vectorized using SIMD instructions, processing many values in parallel.

The key algorithmic insight is a compile-time planning phase that precomputes the exact sequence of byte shuffles, shifts, and masks required for each packing width, making the runtime execution a straight-line sequence of SIMD instructions with no scalar fallback. Thanks to the xsimd abstraction library, this implementation is micro-architecture agnostic: the same algorithm runs efficiently across x86 SSE2, AVX2, and ARM NEON, with more architectures left to be tested.

It yields up to 3x speedup in unpacking, which translates to up to 60% faster Parquet column reads for affected columns.

## Representing Integers

Integer bit packing is the operation of writing an integer on a smaller number of bits than what is required to work with it on a computer.
Today most computers must work with multiples of 8 bits, also called a byte, and integers are represented with a certain number of bytes.
Most performant representations have a capacity of either 32 or 64 bits.
For the sake of simplicity, let us assume that in these bits, non-negative integers are stored in base 2, in increasing base power (little-endian bit order), and padded with zeros for higher unused bits.

For instance the value $13$ can be represented as the following, where contrary to mathematical representation we go from $2^0$ (low order bit) to $2^3$ (high order bit) instead of the opposite, as it will make it easier to work with.
$$13 = 1*2^0 + 0*2^1 + 1*2^2 + 1*2^3 = \underline{1011}_2$$

The following interaction let you explore representation from any number, written over 32 bits.
The green bits are ones with the actual data while the non-colored ones are zero padding to 32 bits.
Padding bits are still valid in the representation, similar to how a number such $13$ can also be written $013$, $0013$, *etc.*
Vertical bars separate bytes from one another.

In [ ]:
#include <clang/Interpreter/CppInterOp.h>

Cpp::AddIncludePath(".");

In [ ]:
#include "blog/all.hpp"

namespace bpacking = arrow::internal::bpacking;

In [ ]:
auto uint_wt = IntegerToolbar();

auto uint_on_change = [](auto value, Uint uint) {
    auto const unpacked_bit_size = static_cast<std::size_t>(uint);
    
    std::cout << "Binary integer representation:\n";
    auto const bit_style = MakeUnpackedColorFn({
        .packed_bit_size = static_cast<std::size_t>(std::bit_width(value)),
        .unpacked_bit_size = unpacked_bit_size,
        .bit_highlight = [](auto) { return false; },
    });
  
    const auto value_le = arrow::bit_util::ToLittleEndian(value);
    PrintBytes(value_le, { .bit_style = bit_style, .max_bytes = unpacked_bit_size / 8 });
};

uint_wt.on_change(uint_on_change);
uint_wt.display();

## Packing Integer Sequences

We can observe that for small numbers, there are few green bits and a lot of redundant zeros.
In general this is not an issue, after all four million such number is roughly 4mB or memory (RAM) used while today's computers typically have several orders of magnitude more available.
In addition, this is the minimal representation needed when making something useful with an integer such as an addition, multiplication, *etc.* (while 8-bit and 16-bit integers are available, applications typically use 32-bit or 64-bit integers).

But the constraints change when we focus on data storage.
Computers can store on disk (SSD) and in the cloud (someone else's disk) much more data than can fit in memory and only access the relevant parts.
There the data stays dormant, taking space often for a long time, and the question of storing only the green part becomes relevant.
In cloud and web environments, moving data over internet is also often a bottleneck.
In this case, reducing the size of the data to transmit leads to great speedups.

Let us see how that would work.

In practice, packing integers more efficiently applies not to single integers, but to sequences of them, such as a column in a database or in a tabular file format like [Apache Parquet](https://parquet.apache.org/).
But if we stored only the relevant bits for each value, the number of relevant bits would vary, making it impossible to tell where one ends and the next begins.
We therefore want to pack them in common sizes, and the first step is to find what size is needed to fit all our values.
The following function `MaxBitWidth` does just that.

In [ ]:
template <typename Range>
auto MaxBitWidth(Range&& in) -> std::size_t {
    auto bit_width = [](auto x) { return std::bit_width(x); };
    const auto value = std::ranges::max(in, {}, bit_width);
    return static_cast<std::size_t>(bit_width(value));
}

Then, we will represent all our values with that common number of bits, setting unused bits to zero, and concatenate them into a dense sequence.
The following `PackExact` is a naive implementation of such a routine, writing bits one by one to the output sequence.
To understand the following recall that the minimum entity we can address (work with) on a typical computer is a byte (8 bits).
We use a byte buffer (`std::vector<std::uint8_t>` here) to hold the output, the shift operation (`<<` and `>>`) to position bits within a byte, AND operations (`&`) with a mask to isolate a specific bit from each input integer, and OR operations (`|`) to write that bit's value into the zero-initialized output byte.

In [ ]:
/// Naively pack integers into bit-packed format.
template <typename Range>
  requires std::ranges::input_range<Range>
auto PackExact(Range&& in, int packed_bit_width) -> std::vector<std::uint8_t> {
  const int batch_size = static_cast<int>(in.size());

  // Calculate required output size in bytes
  const int output_bytes = arrow::bit_util::BytesForBits(batch_size * packed_bit_width);
  auto out = std::vector<std::uint8_t>(output_bytes, std::uint8_t{0});

  // Pack each value bit by bit
  int out_bit_idx = 0;
  for (const auto val : in) {
    // Write each bit of this value
    for (int bit_idx = 0; bit_idx < packed_bit_width; bit_idx++) {
      const int out_byte_idx = out_bit_idx / 8;
      const int out_bit_offset_in_byte = out_bit_idx % 8;

      // Extract bit in position bit_idx and move it to the first bit
      constexpr std::uint8_t kFirstBitMask = 0b00000001;
      const std::uint8_t bit = (val >> bit_idx) & kFirstBitMask;
      // Write bit in output byte at correct position
      out[out_byte_idx] |= (bit << out_bit_offset_in_byte);

      out_bit_idx++;
    }
  }

  return out;
}

The following interactive widget shows the initial values (unpacked) and bit-packed values for a given sequence.

In [ ]:
auto packing_wt = SequenceToolbar();

auto packing_on_change = []<typename Widget>(Widget const& w){
    using value_type = typename Widget::value_type;
    auto const values = w.values();

    const auto unpacked_bit_size = static_cast<std::size_t>(w.unpacked_uint_max_bits());
    const auto packed_bit_size = static_cast<std::size_t>(w.packed_bit_size());
    const auto bytes = PackExact(values, packed_bit_size);
    const auto current_val = w.step() % values.size();

    std::cout << "Storing " << values.size() << " unpacked values over "
        << unpacked_bit_size << " bits each takes a total of "
        << sizeof(value_type) * values.size() << " bytes:\n";

    // Abusing PackExact to dynamically static cast to the input unpacked type
    const auto uint_bytes = PackExact(values, unpacked_bit_size);
    PrintBytes(uint_bytes, {
        .lane_bit_size = unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakeUnpackedColorFn({
            .packed_bit_size = packed_bit_size,
            .unpacked_bit_size = unpacked_bit_size,
            .selected_value_idx = current_val,
        }),
    });

    std::cout << "\nStoring the same " << values.size()
        << " values packed in " << packed_bit_size << " bits each takes a total of "
        << bytes.size() << " bytes:\n";
    
    PrintBytes(bytes, {
        .lane_bit_size = unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakePackedColorFn({
            .packed_bit_size = packed_bit_size,
            .selected_value_idx = current_val,
            .n_valid_bits = values.size() * packed_bit_size,
        }),
    });
};

packing_wt.on_change(std::move(packing_on_change));
packing_wt.display();

As we can see, for a sequence of small values we can save a lot of space.
How relevant is this?
Well a lot of numbers used by human are generally rather small: number of children, price (in cents) *etc*.
Categories are also encoded as small integer.
In Parquet files this is part of *dictionnary encoding*.
For example, let us imagine a user choice between three frequency of notifications.
This can be represented as:
$$None \rightarrow 0$$
$$Essentials \rightarrow 1$$
$$All \rightarrow 2$$
In this case the space reduction is significant.

In addition, in Parquet files, columns with list types are encoded using repetition and definition levels, from
Google's [Dremel paper](https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/36632.pdf). For example, the following column:

| basket |
| --- |
| ["Apple", "Banana"] |
| ["Banana", "Cookies", "Juice"] |

is flattened and stored alongside a repetition level column:

| basket.value | basket.repetition |
| --- | --- |
| "Apple" | 0   |
| "Banana" | 1   |
| "Banana" | 0   |
| "Cookies" | 1   |
| "Juice" | 1   |

where 0 means "this value starts a new list", and 1 means "this value continues the current list". For a list of lists, a level of 2 would mean "this value continues the current inner list".
While the full repetition and definition level scheme is beyond the scope of this blog post, the key point is that these levels are bounded by the depth of nesting, known from the column type.
This is therefore another example of small integers appearing frequently in Parquet files, beyond just the column values themselves.

Note that a single large value can completely eliminate the space saving by forcing all values in the sequence to be written with a large number of mostly-zero bits.
In practice, packing is therefore done in *runs*, for instance, 1024 values at a time, each using its own minimum bit width.

That said, packing integers is a tradeoff.
While it reduces storage size, reading packed data requires unpacking it, which is far more computationally expensive than simply copying values stored as plain 32- or 64- bit integers.
Instead, we now have to shift and mask bytes, handle values split across bytes, memory cache lines *etc*.

This tradeoff is particularly important because large Parquet datasets are read far more than they are written.
They are queried repeatedly by business analysts, data scientists, and machine learning pipelines in modern
[data lakehouses](https://www.databricks.com/glossary/data-lakehouse).
Speeding up the integer unpacking routine is therefore crucial and was the focus of our [recent contribution to Apache Arrow](TODO: link to Post 1).

## Unpacking One Value at a Time

As noted, the `PackExact` function we used earlier has awful performance: by writing bits one by one, it unnecessarily manipulates all eight bits of a byte, and that for every single bit it needs to write.

A naive unpacking routine would have the same pitfalls. We want to avoid bit-wise operations, wastefully touching 8 bits to read just 1, but the challenge is that we can only read memory in multiples of 8 bits, aligned to byte boundaries.
Since the value of interest will not necessarily start on a byte boundary, the solution is to shift it into alignment, then mask out the surrounding bits from adjacent values using the known bit width of that run, and finally read.
The following interactive widget illustrates the algorithm.

In [ ]:
auto unpacking_wt = SequenceToolbar({.interval_ms = 1000});

auto unpacking_on_change = []<typename Widget>(Widget const& w){
    using value_type = typename Widget::value_type;
    const auto values = w.values();
    const auto packed_bit_size = static_cast<std::size_t>(w.packed_bit_size());

    const auto params = UnpackExactSelectedParams{
        .value_idx = w.step() % values.size(),
        .n_values = values.size(),
        .packed_bit_size = packed_bit_size,
        .unpacked_bit_size = static_cast<std::size_t>(w.unpacked_uint_max_bits()),
        .max_spread_bytes = static_cast<std::size_t>(bpacking::PackedMaxSpreadBytes(packed_bit_size, 0)),
    };
    const auto bytes = PackExact(values, params.packed_bit_size);
    // Reproducing state of previous step
    auto out_values = std::decay_t<decltype(values)>(values.size(), value_type{0});
    std::copy(values.cbegin(), values.cbegin() + params.value_idx, out_values.begin());

    std::cout << "Input packed values:\n";
    PrintBytes(bytes, {
        .bit_style = MakePackedColorFn({
            .packed_bit_size = params.packed_bit_size,
            .selected_value_idx = params.value_idx,
            .n_valid_bits = values.size() * params.packed_bit_size,
        }),
    });

    std::cout << "\nSelect memory region:\n    ";
    PrintPackedSelect(bytes, params);

    auto buffer = std::uint64_t{};
    auto n_bytes_to_copy = std::min(
        params.max_spread_bytes,
        bytes.size() - params.ValueByteStart()
    );
    std::memcpy(&buffer, bytes.data() + params.ValueByteStart(), n_bytes_to_copy);
    buffer = arrow::bit_util::FromLittleEndian(buffer);
    std::cout << "\nCopy the current value and extra bits:\n"
        << "    read(start=" << params.ValueByteStart() << " ,size="
        << n_bytes_to_copy <<")\n    ──► ";
    PrintBufferSelect(buffer, params);

    buffer >>= params.SpreadShiftBits();
    std::cout << "\nShift buffer to align current value:\n    ";
    PrintBufferSelect(buffer, params);  
    std::cout << " << " << params.SpreadShiftBits() << "\n    ──► ";
    PrintShiftedBuffer(buffer, params);
   
    auto const mask = arrow::bit_util::LeastSignificantBitMask<std::uint64_t>(params.packed_bit_size);
    buffer = buffer & mask;
    std::cout << "\nApply bit mask:\n        ";
    PrintShiftedBuffer(buffer, params);
    std::cout << "\n     &  ";
    PrintMask(mask, params);    
    std::cout << "\n    ──► ";
    PrintUnpackedSelect(buffer, params);
    
    out_values.at(params.value_idx) = buffer;
    std::cout << "\nSave to ouput:\n";
    // Abusing PackExact to dynamically static cast to the input unpacked type
    const auto uint_bytes = PackExact(out_values, params.unpacked_bit_size);
    PrintBytes(uint_bytes, {
        .lane_bit_size = params.unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakeUnpackedColorFn({
            .packed_bit_size = params.packed_bit_size,
            .unpacked_bit_size = params.unpacked_bit_size,
            .selected_value_idx = params.value_idx,
            .n_colored_values = params.value_idx + 1,
        }),
    });
};

unpacking_wt.on_change(std::move(unpacking_on_change));
unpacking_wt.display();

A subtlety of this is that the number of bytes to read per value depends on the packed bit size:
If you look carefully at the algorithm, a packed bit size of 2 or 4 requires reading only one byte at a time to extract the value; for packed bit sizes of 10 or 12, two bytes are needed.
This is simply the smallest number of bytes that can fully contain the given number or bits,
or $\left\lceil{\frac{bit{\_}size}{8}}\right\rceil$.

However, that is not what happens when the packed bit size is 3, 5, 6, 7, 11 *etc*.
Now, although a 3-bit packed size can fit in one byte, some values inevitably cross a byte boundary when packed together (in this case the third, for example), and spread over two bytes.
This means we need to extract $\left\lceil{\frac{bit{\_}size}{8}}\right\rceil + 1$ bytes from memory to have the value in full.
This is captured by the function `PackedMaxSpreadBytes` from our contribution which is central to dimensioning our unpacking algorithm.
For a packing size of 59 bits, for instance, values can spread across up to 9 bytes.
This is larger than the largest unsigned integer type available to us (`uint64_t` is 8 bytes).
We call this case *large* and need to handle it separately.

This unpacking algorithm is implemented in our recent contribution to Apache Arrow as the [`unpack_exact` function](https://github.com/apache/arrow/blob/main/cpp/src/arrow/util/bpacking_dispatch_internal.h).

The function is more complex than what is described above; here are the differences of interest:

First it is *exact*: it unpacks precisely the number of values requested, and can resume from a non-byte-aligned position.
For example, after extracting two 3-bit values, the next will start at bit 6, which is not a begining of a byte.
The fuction handles this to remain general and composable.

Second, both the packed bit size and the unpacked integer type are template parameters known at compile time, a C++ template metaprogramming technique.
This enables several optimizations: the size of the unsigned integer reading buffer can be adapted to the packed bit size (types need to be decided at compile time in C++), and values like the bit mask can be hardcoded, making them visible to the compiler for further optimization.
In practice, however, the packed bit size is only known at runtime, as it is stored in the Parquet file's encoding metadata.
Since there are only finitely many possible packed bit sizes (up to 64 for `uint64_t`), the solution is to compile the function for each of them and at runtime *jump* to the correct one.
This is what `unpack_jump` does, a pattern known as *microkernel dispatching* commonly used in high performance scientific computing.

Is the complexity introduced by this metaprogramming technique really necessary?
For `unpack_exact` alone, the performance gains are modest, but the technique supports a SIMD-accelerated variant processes multiple packed integers at once yielding much larger improvements.

To see why this matters, consider three inefficiencies of the naive implementation, processing one packed integer at a time:
First, fetched bytes for one packed value often contain bits for other values as well, which are discarded and re-fetched later; for a 2-bit packing width, each byte holds 4 values and is fetched up to four times.
Second, a single larger fetch could cover multiple values at once; 8 bytes would yield 32 2-bit-packed integers in one go, sharing the fetch cost across all of them.
Third, each value is extracted independently using several instructions (shifts, masks, ORs), even though dedicated CPU pathways exist that can perform such operations on multiple values simultaneously.
While the first two inefficiencies are partially mitigated by CPU caching and compiler optimizations, the third is not — and it is what motivated our [contribution to Apache Arrow](https://github.com/apache/arrow/pull/47994). By leveraging [SIMD (*Single Instruction, Multiple Data*)](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data) instructions, which provide dedicated hardware pathways for processing multiple values in parallel, we load a wide range of bytes and extract multiple packed integers in a single operation. The following sections explain this in more detail.

## Planning at Compile Time

### SIMD Primer

SIMD instructions operate on *registers*, fast, fixed-size memory slots built into the CPI, processing all their contents in a single instruction.
They were introduced to accelerate workloads that apply the same operation repeatedly to large arrays of data, such as encountered in signal processing, graphics and scientific computation, and precisely the class of problem we are dealing with when unpacking integers.

Which SIMD instructions are available depends on both the CPU architecture and the specific CPU model (its micro-architecture) as within the same architecture not all CPUs support the same standards: On [x86](https://en.wikipedia.org/wiki/X86) (most current Intel & AMDs, but not the recent Apple M-series), [SSE](https://en.wikipedia.org/wiki/Streaming_SIMD_Extensions) was introduced in 1999 and SSE2 in 2000; both provide 128-bit registers and are universally available on all modern x86 CPUs, as they became mandatory with the 64-bit x86 architecture. [AVX](https://en.wikipedia.org/wiki/Advanced_Vector_Extensions), introduced in 2011, doubles this to 256-bit registers but is only available on more recent CPU models.
[ARM-based](https://en.wikipedia.org/wiki/ARM_architecture_family) CPUs, such as the Apple M-series, use a different SIMD instruction set called [NEON](https://en.wikipedia.org/wiki/ARM_architecture_family#Advanced_SIMD_(Neon)) which is not directly compatible with SSE or AVX code.

In practice, SIMD instructions are accessed through dedicated types and functions in C++.
For example, on a CPU with NEON, a `uint32x4_t` is a SIMD-enabled array of four `std::uint32_t`, and two such arrays can be summed in one SIMD-instruction with the `vaddvq_u32` function.
The CPU will execute this dedicated instruction in a single shot, avoiding the loop required when using `std::array<std::uint64_t, 4>`.

Where the naive algorithm already required computing several intermediary values (byte offsets, shifts, and masks), unpacking multiple values at once as in the SIMD variant requires even more.
These constants depend on the target integer size, the SIMD register size, and the packing bit width, all of which are known at compile time thanks to the previously described template metaprogramming.
At first, it seems that some constant can be computed from these through closed-form expressions, such as in some cases `(1 << n_bits) - 1` for a bit mask with `n_bits`.
But with increasing complexity, finding such closed-form expressions becomes difficult as the previously discussed `PackedMaxSpreadBytes` example showed.
This is where C++ `constexpr`, a full programming language, becomes essential because it allows computing them at compile time.
In a sense, this precomputation phase, which we call *plan building* (a *plan* contains all the compile-time constants needed for unpacking such as byte-offsets, shifts, masks, and more), works by *simulating* the described unpacking algorithm at compile time, but recording the intermediary values later to be used as constants instead of running on actual data.
Microkernel dispatching then ensures that at runtime, the correct specialized function, with all its precomputed constants, is selected for the actual packing bit width read from the Parquet file.

Using SIMD and microkernel dispatching for unpacking integers is not new.
Our algorithm is based on work from Daniel Lemire and Leonid Boytsov,
[*Decoding billions of integers per second through vectorization*](http://arxiv.org/abs/1209.2137), made available in the [LittleIntPacker library](https://github.com/fast-pack/LittleIntPacker/blob/master/src/horizontalpacking32.c).
We extended it to more integer types (we handle `uint8/16/32/64` on SSE2, AVX, NEON and WASM SIMD, while SVE and AVX-512 aren't tested yet due to lack of hardware. LittleIntPacker handles only `uint32` on SSE2, and Arrow's old implementation was also missing some), as well as to support varying SIMD register sizes.
Our second contribution is to leverage C++ template and `constexpr` metaprogramming to implement the algorithm in a more unified way:
Both LittleIntPacker and Arrow's previous implementation relied on a Python script using string templates to generate the required C++ constants at compile time, an approach that worked but introduced a hard split between the code generator and the generated code. In Apache Arrow, for example, 40,000 lines of C++ code were generated and committed into the codebase, forcing developers to reason across two languages and deal with a separate generation step rather than with the algorithm directly.
Our approach replaces this with about 1,000 lines of extensively-commented C++ code, and expresses the entire algorithm, from compile-time plan building to runtime unpacking, uniformly in a single language, integrated into the Arrow C++ codebase.
While this may seem to be an implementation detail, it makes the relationship between precomputed constants and the unpacking logic much more explicit and maintainable.

### Unpacking a Single SIMD Register

To understand the SIMD algorithm, let's restate our goal: read packed values from a contiguous input array, and write unpacked values to a contiguous output array, using only SIMD registers since moving data between SIMD and scalar registers is costly.
On the input side, we load *one* SIMD register with n *packed* values from memory.
On the output side, we produce several SIMD registers with n *unpacked* values, writing each to memory as it's produced.
For example, with 12-bit values unpacked to 32-bit integers on a 128-bit register, we load one register containing 8 packed values (96 bits, with 32 bits unused) and produce two output registers with 8 unpacked values (4 per register).

We then need to go from input to output using nothing but SIMD instructions—standard operations, but on vectors.
If we had to perform the complex bit operations that we previously described, this would be a formidable task.
Metaprogramming comes to our rescue: template metaprogramming gives compile-time access to the base parameters (packed bit size, SIMD register size, etc.), and `constexpr` computes a *plan*—a data structure that converts the bit indices of packed values into arrays of byte ranges, permutation indices, and shift amounts that we can apply directly with SIMD instructions.
Functions are compiled for each possible bit width, composing the correct sequence of instructions for each parameter combination. At runtime, dynamic dispatch selects the appropriate function based on the bit width found in the Parquet file.
Metaprogramming here is not a stylistic choice—it's what makes efficient SIMD possible.

To simplify understanding of the following, let's view each output SIMD register as divided into *slots*, one per unpacked value in that register.
For example, unpacking to 32-bit integers on a 128-bit register yields four slots per register, each slot holding one 4-byte value. The algorithm proceeds in three steps:

First, a **swizzle** operation (also called shuffle or permute in other contexts) places the *bytes* containing each packed value within its respective output slot, establishing the correct order.

Second, a **shift** operation aligns the value's *bits* to the start of the slot.

Third, a **mask** zeroes out the unwanted bits, leaving only the unpacked value.

An important optimization reduces the number of swizzle operations (otherwise one per output register), which matters since swizzle is computationally expensive compared to other SIMD operations.
When the byte spread is smaller than the slot width, multiple packed values can share the same slot, allowing multiple registers to be folded into one *swizzle register* served by a single swizzle.
In this case, each output register is produced by a separate shift applied to the same swizzle register, with the shift amount accounting for both the bit offset of the value within its bytes and the byte offset introduced by the folding.
For instance, with a 3-bit packing width, the byte spread is 2, and assuming each output slot is 4 bytes wide, two values fit per slot: the first output register's values occupy bytes 0-1 of each slot, and the second output register's values occupy bytes 2-3.
With a 2-bit packing width, the byte spread is 1, so four values fit per slot, and one swizzle serves four output registers.
For larger packed bit sizes like 13 bits, the byte spread is three, leaving only one byte unused per slot, which is not enough to fit another value, and one swizzle serves only one output register.

We call this the *medium* plan because the byte spread fits within an output slot (spread ≤ slot size).
All our examples so far (3-bit, 13-bit, 17-bit unpacked to `uint32_t`) are medium plans since their byte spreads fit within the 4-byte output slot.
When the byte spread exceeds the slot size, for example, unpacking 15-bit values to `uint16_t` where the 3-byte spread exceeds the 2-byte slot, a different *large* plan is needed, which we discuss later.

The following interactive widget shows how packed input values are extracted with the medium plan.

In [ ]:
auto MakeSimdInput(auto const& plan) {
    auto const& shape = plan.kShape;
    auto const n_values = static_cast<std::size_t>(plan.unpacked_per_swizzle());
    
    auto const alpha_bits = MakeAlphaValues({
        .packed_bit_size = static_cast<std::size_t>(shape.packed_bit_size()),
        .n_values = n_values,
        .total_bits = static_cast<std::size_t>(shape.simd_bit_size()),
    });

    auto const palette = MakePaletteStyle(LIGHT_ORANGE, 3 * n_values / 4 + 4);
    auto packed_bit_style = MakePackedColorFn({
        .packed_bit_size = static_cast<std::size_t>(shape.packed_bit_size()),
        .n_valid_bits = n_values * static_cast<std::size_t>(shape.packed_bit_size()),
        .bit_highlight = [](auto) { return false; },
        .styles = palette,
    });

    auto styles = std::vector<TextStyle>();
    for(std::size_t k = 0; k < alpha_bits.size(); ++k){
        styles.push_back(packed_bit_style(k));
    }

    return std::pair(alpha_bits, styles);
}

auto SwizzleInput(
    auto const& plan, auto const& input, auto const& input_style,
    int read_idx = 0, int swizzle_idx = 0
) {
    auto const& shape = plan.kShape;
    auto const& swizzles = plan.swizzles.at(read_idx).at(swizzle_idx);
    auto const& all_shifts = plan.shifts.at(read_idx).at(swizzle_idx);
    
    auto swizzled = std::vector<char>(input.size(), '0');
    auto swizzled_style = std::vector<TextStyle>(input.size());
    
    for(int out_bit_idx = 0; out_bit_idx < input.size(); ++out_bit_idx){
        auto const bit_offset = out_bit_idx % kBitsPerBytes;
        auto const out_byte_idx = out_bit_idx / kBitsPerBytes;
        auto const in_byte_idx = swizzles.at(out_byte_idx);
        auto const in_bit_idx = in_byte_idx * kBitsPerBytes + bit_offset;

        // Larger indices are used to explicitly set to zero
        if (in_byte_idx >= shape.simd_byte_size()){
            continue;
        }
        swizzled.at(out_bit_idx) = input.at(in_bit_idx);
    
        auto const out_unpacked_idx = out_bit_idx / shape.unpacked_bit_size();
        auto const out_unpacked_bit_offset = out_bit_idx % shape.unpacked_bit_size();

        // Here we use the computed shifts to reverse the computation of where the value
        // needs to be colored, but in the regular algorithm , we compute where the value
        // falls in order to get the shifts.
        auto style = TextStyle{.fg = BLACK, .bg = LIGHT_GREY};
        for(auto const& shifts : all_shifts) {
            auto const shift = shifts.at(out_unpacked_idx);
            if (out_unpacked_bit_offset >= shift && out_unpacked_bit_offset < shift + shape.packed_bit_size()){
                style = input_style.at(in_bit_idx);
            }
        }
        swizzled_style.at(out_bit_idx) = style;
    }

    return std::pair(swizzled, swizzled_style);
}

auto ShiftSwizzled(
    auto const& plan, auto const& swizzled, auto const& swizzled_style,
    int read_idx = 0, int swizzle_idx = 0, int shift_idx = 0
) { 
    auto const& shape = plan.kShape;
    auto const& shifts = plan.shifts.at(read_idx).at(swizzle_idx).at(shift_idx);
    
    auto shifted = std::vector<char>();
    auto shifted_style = std::vector<TextStyle>();
    
    for(int unpacked_idx = 0; unpacked_idx < plan.unpacked_per_shift(); ++unpacked_idx){
        auto const unpacked_bit_idx = shape.unpacked_bit_size() * unpacked_idx;
        auto const val_bit_start = unpacked_bit_idx + shifts.at(unpacked_idx); 
        auto const unpacked_bit_end = unpacked_bit_idx + shape.unpacked_bit_size();
        
        shifted.insert(shifted.end(), swizzled.cbegin() + val_bit_start, swizzled.cbegin() + unpacked_bit_end);
        shifted.resize((unpacked_idx + 1) * shape.unpacked_bit_size(), '0');
        shifted_style.insert(shifted_style.end(), swizzled_style.cbegin() + val_bit_start, swizzled_style.cbegin() + unpacked_bit_end);
        shifted_style.resize((unpacked_idx + 1) * shape.unpacked_bit_size());
    }

    return std::pair(shifted, shifted_style);
}

auto MaskRegsiter(auto const& plan, auto const& shifted, auto const& shifted_style) { 
    const auto shape = plan.kShape;
    
    auto masked = shifted;
    auto masked_style = shifted_style;
    
    for(int k = 0; k < masked.size(); ++k){
        if (k % shape.unpacked_bit_size() >= shape.packed_bit_size()){
            masked.at(k) = '0';
            masked_style.at(k) = {};
        }
    }

    return std::pair(masked, masked_style);
}

template<typename Plan>
  requires (Plan::kShape.is_medium())
auto PrintSimdExample(Plan const& plan) {
    const int read_idx = 0;
    const int swizzle_idx = 0;
    
    const auto shape = plan.kShape;

    auto const [input, input_style] = MakeSimdInput(plan); 
    std::cout << "Input register showing the " << plan.unpacked_per_swizzle()
        << " values extracted in this iteration:\n";
    PrintValuesAsBits(input, {
        .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),
        .lane_end = " ",
        .bit_style = MakeColorFn(input_style),
    }); 

    auto const [swizzled, swizzled_style] = SwizzleInput(plan, input, input_style, read_idx, swizzle_idx);  
    std::cout << "\nSwizzled register:\n";
    PrintValuesAsBits(swizzled, {
        .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),
        .lane_end = " ",
        .bit_style = MakeColorFn(swizzled_style),
    });

    for(int shift_idx = 0; shift_idx<plan.shifts.at(0).at(0).size(); ++shift_idx){
        auto const& [shifted, shifted_style] = ShiftSwizzled(plan, swizzled, swizzled_style, read_idx, swizzle_idx, shift_idx);   
        std::cout << "\n ─► Shifted register " << shift_idx << ":\n    ";
        PrintValuesAsBits(shifted, {
            .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),   
            .lane_end = " ",
            .bit_style = MakeColorFn(shifted_style),
        });
    
        auto const [masked, masked_style] = MaskRegsiter(plan, shifted, shifted_style);    
        std::cout << "\n    Masked register " << shift_idx << ":\n    ";
        PrintValuesAsBits(masked, {
            .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),
            .lane_end = " ",
            .bit_style = MakeColorFn(masked_style),
        });
    }
}

template <int kPackedBitSize, Uint kUint>
struct PrintSimdExampleDispatcher {
    /// Hard coded for visualization
    static constexpr int kSimdBitSize = 128;
    using uint_t = UintType<kUint>;
    using Traits = bpacking::KernelTraits<uint_t, kPackedBitSize, kSimdBitSize>;

    static void call(){
        if constexpr (Traits::kShape.is_medium()) {
            PrintSimdExample(bpacking::MediumKernelPlan<Traits, {32}>::Build());
        } else {
            throw std::runtime_error("The packed bit sizes are not compatible with medium plans."); 
        }
    }
};

In [ ]:
auto swizzle_wt = IntegerToolbar({
    .number_description = "Packed bit size",
    .number_min = 1,
    .uint_description = "Unpacked type",
});

auto swizzle_on_change = [](auto value, Uint uint) {   
    Dispatch<PrintSimdExampleDispatcher>(static_cast<int>(value), uint);
};

swizzle_wt.on_change(swizzle_on_change);
swizzle_wt.display();

While a single swizzle/shift/mask sequence efficiently extracts values, it typically doesn't cover a full input register.
For example, when unpacking 3-bit packed values (byte spread 2) to `uint32_t` on 128-bit SIMD registers (4 slots per register), one swizzle operation only processes 8 values (24 bits = 3 bytes) out of the 16 bytes available in the input register.
To fully utilize the input, the algorithm must execute multiple swizzles, each covering a different byte range.
We will see how these operations are combined into a *microkernel* in the next section.

### SIMD Unpacking Loop

A key constraint arises from byte alignment.
When the packed bit size is not a multiple of 8, the bit position after a swizzle may not be byte-aligned.
For example, when unpacking 13-bit values to `uint32_t` on 128-bit registers, one swizzle processes 4 values (52 bits), leaving the next value starting at bit 52 — byte 6, bit 4.
A second swizzle processes 4 more values (bits 52–103), with its own precomputed swizzle indices (referencing bytes 6–12) and shift amounts (accounting for each value's bit position within those bytes).
After 8 values total (104 bits = 13 bytes), we return to byte alignment.

The plan doesn't necessarily consume all bytes from a register load.
In the 13-bit example, two swizzles process 104 bits (13 bytes), leaving 24 bits unused in the 128-bit register.
The next plan invocation simply loads a fresh register starting at byte 13, re-reading those bytes.
This overlap is intentional: we have no choice if we want byte alignment at plan boundaries, and the re-read bytes are likely already in cache from the previous load.

A second constraint arises when a single read cannot reach byte alignment.
For example, when unpacking 17-bit values, one swizzle processes 4 values (68 bits), ending at bit 68 (bit 4 of byte 8).
To reach byte alignment, the algorithm performs a second read from byte 8 of the input array into a new SIMD register and processes 4 more values. The 4-bit offset is precomputed into the plan's shift amounts for this read.
After 8 values total (136 bits = 17 bytes), we return to byte alignment, and we can reexecute the kernel from the start.

The *plan* is the compile-time data structure that encodes all of this: the byte offsets for each input read, the byte permutation patterns for each swizzle (called *swizzle indices*), and the per-slot shift amounts.
The number of values per plan is computed to ensure the total bits processed is a multiple of 8, guaranteeing byte alignment at plan boundaries.

The *microkernel* is the function that executes one plan — chaining the necessary reads, swizzles, and shifts to unpack a fixed number of values while ending byte-aligned.
It is compiled separately for each combination of packed bit size, output integer type (and SIMD register size if compiled for multiple microarchitectures).

The outer runtime loop invokes the microkernel repeatedly, advancing through the input array.
Each invocation unpacks a fixed number of values (e.g., 8 values for both 13-bit and 17-bit examples) and advances the input pointer by the corresponding number of bytes (13 or 17 bytes respectively).
The loop exits when either not enough values remain to fill a plan, or when continuing would read past the input buffer.
Because each plan ends byte-aligned, no bit offset tracking is needed between iterations.

When the input doesn't start byte-aligned (rare in practice), a *prolog* phase extracts values one-by-one using the scalar method described earlier until reaching a byte boundary, then hands off to the SIMD loop.
Similarly, an *epilog* phase handles any remaining values that don't fill a complete plan.

Compile-time computation is not incidental here—it is essential for efficient SIMD.
The inner loops within the microkernel (iterating over reads, swizzles, and shifts) are fully unrolled at compile time using C++17 fold expressions, eliminating loop overhead and enabling better instruction scheduling.
More importantly, the swizzle indices and shift amounts are `constexpr` data, converted to compile-time constants via template metaprogramming.
This allows the compiler to hoist them into registers before the runtime loop, reusing them in every iteration with no runtime index computation.

The following interactive example shows the concrete plan structure, that is the reads, swizzles, and shift amounts that the microkernel executes.
Many times, some loops will have only one iteration.

In [ ]:
template<typename Plan>
  requires (Plan::kShape.is_medium())
auto PrintSimdPlan(Plan const& plan) {  
    std::cout << "Medium plan for unpacking " << plan.kShape.packed_bit_size()
    << " bits to " << plan.kShape.unpacked_bit_size()
    << " bits using " << plan.kShape.simd_bit_size()
    << " bits SIMD registers:\n";
    std::cout << "   • Unpacked values per μKernel : " << plan.unpacked_per_kernel() << "\n";
    std::cout << "   • Unpacked bytes per μKernel  : " << plan.total_bytes_read() << "\n";

    std::cout << "Steps:\n";
    for (int r = 0; r < plan.reads.size(); ++r) {
        std::cout << "  ─► Read at offset: " << plan.reads.at(r) << "\n";
      
        for (int sw = 0; sw < plan.swizzles.at(r).size(); ++sw) {
            std::cout << "       ─► Swizzle with indices: ";
            PrintJoinedValues(plan.swizzles.at(r).at(sw));
            std::cout << "\n";
            
            for(int sf = 0; sf < plan.shifts.at(r).at(sw).size(); ++sf){
                std::cout << "            ─► Right shift by: ";
                PrintJoinedValues(plan.shifts.at(r).at(sw).at(sf));
                std::cout << "\n";
            }
        }
    }
}

template <int kPackedBitSize, Uint kUint>
struct PrintSimdMediumPlanDispatcher {
    /// Hard coded for visualization
    static constexpr int kSimdBitSize = 128;
    using uint_t = UintType<kUint>;
    using Traits = bpacking::KernelTraits<uint_t, kPackedBitSize, kSimdBitSize>;

    static void call(){
        if constexpr (Traits::kShape.is_medium()) {   
            PrintSimdPlan(bpacking::MediumKernelPlan<Traits, {32}>::Build());
        } else {
            std::cerr << "The packed bit size " << kPackedBitSize << " is too large for large plans.\n";
        }
    }
};

In [ ]:
auto medium_plan_wt = IntegerToolbar({
    .number_description = "Packed bit size",
    .number_min = 1,
    .uint_description = "Unpacked type",
});

auto medium_plan_on_change = [](auto value, Uint uint) {   
    Dispatch<PrintSimdMediumPlanDispatcher>(static_cast<int>(value), uint);
};

medium_plan_wt.on_change(medium_plan_on_change);
medium_plan_wt.display();

### The Large Plan

If you tried different combinations in the previous interactive example, you may have noticed that some configurations don't work, larger packing sizes specifically.
This happens when the byte spread exceeds the output slot size.
For example, unpacking 15-bit values into `uint16_t` requires reading 3 bytes per value, but each output slot is only 2 bytes wide.
We call this a *large* plan, in contrast to the *medium* plans we've seen so far.

The core problem is that a single swizzle cannot place all the bytes for a value into its output slot because there simply isn't enough room.
The solution is to split the work into two passes.
The first pass swizzles and shifts the bytes that fit in the slot (the "low" part of each value).
The second pass swizzles and shifts the overflow bytes (the "high" part), positioning them to align with what the first pass produced.
We then merge the two partial results with bitwise OR (`vpor` on x86, `orr` on ARM).

One simplification: because large plans only occur with larger packing sizes, the medium plan's nested loops (multiple swizzles per read, multiple shifts per swizzle) collapse to exactly one swizzle and one shift per pass.

The following widget lets you print such large plans.

In [ ]:
template<typename Plan>
  requires (Plan::kShape.is_large())
auto PrintSimdPlan(Plan const& plan) {
    std::cout << "large plan for unpacking " << plan.kShape.packed_bit_size()
    << " bits to " << plan.kShape.unpacked_bit_size()
    << " bits using " << plan.kShape.simd_bit_size()
    << " bits SIMD registers:\n";
    std::cout << "   • Unpacked values per μKernel : " << plan.kPlanSize.unpacked_per_read() << "\n";
    std::cout << "   • Unpacked bytes per μKernel  : " << plan.total_bytes_read() << "\n"; 

    std::cout << "Steps:\n";
    for (int r = 0; r < plan.kPlanSize.reads_per_kernel(); ++r) {
        std::cout << "  ─► Read at offset: " << plan.reads.at(r);
        std::cout << "\n       ─► Swizzle for low part with indices : ";
        PrintJoinedValues(plan.low_swizzles.at(r));
        std::cout << "\n       ─► Right shift for low part by : ";
        PrintJoinedValues(plan.low_rshifts.at(r));
        std::cout << "\n       ─► Swizzle for high part with indices: ";
        PrintJoinedValues(plan.high_swizzles.at(r));
        std::cout << "\n       ─► Right shift for high part by: ";
        PrintJoinedValues(plan.high_lshifts.at(r));
        std::cout << "\n       ─► Merge high and low parts with bitwise OR\n";
    }
}

template <int kPackedBitSize, Uint kUint>
struct PrintSimdLargePlanDispatcher {
    /// Hard coded for visualization
    static constexpr int kSimdBitSize = 128;
    using uint_t = UintType<kUint>;
    using Traits = bpacking::KernelTraits<uint_t, kPackedBitSize, kSimdBitSize>;

    static void call(){
        if constexpr (Traits::kShape.is_medium()) {   
            std::cerr << "The packed bit size " << kPackedBitSize << " is too small for large plans.\n";
        } else if constexpr (Traits::kShape.is_large()) {
            PrintSimdPlan(bpacking::LargeKernelPlan<Traits>::Build());
        } else if constexpr (Traits::kShape.is_oversized()) {
            std::cerr << "The packed bit size " << kPackedBitSize << " is too large for large plans.\n";
        }
    }
};

In [ ]:
auto large_plan_wt = IntegerToolbar({
    .number_description = "Packed bit size",
    .number_min = 1,
    .number_default = 27,
    .uint_description = "Unpacked type",
});

auto large_plan_on_change = [](auto value, Uint uint) {   
    Dispatch<PrintSimdLargePlanDispatcher>(static_cast<int>(value), uint);
};

large_plan_wt.on_change(large_plan_on_change);
large_plan_wt.display();

## Writing SIMD kernels

### Generic SIMD and fallbacks with xsimd

We've now seen how the algorithm works: the plan structure, the assembly it generates, and why it's fast.
But we glossed over an important detail: how do we actually *write* portable C++ code that compiles to that assembly across different architectures?
It's not as straightforward as calling SIMD intrinsics directly.

The challenge is that SIMD intrinsics differ across architectures.
[x86](https://en.wikipedia.org/wiki/X86) and [ARM](https://en.wikipedia.org/wiki/ARM_architecture_family) don't just use different function names, they offer different operations with different semantics.
For Python readers, imagine if [NumPy](https://numpy.org/) and [PyTorch](https://pytorch.org/) had completely incompatible APIs, and you had to write code that worked with both.
To make matters worse, x86 has evolved over decades, and newer instruction sets (SSE, AVX, AVX2, AVX-512) don't always follow consistent conventions.
Some generations lack basic operations (like per-element shifts) for certain data types.

The [xsimd](https://github.com/xtensor-stack/xsimd) library, already used in Apache Arrow, abstracts over these differences.
It provides a uniform interface for SIMD programming, independent of the target architecture.
Beyond simple function aliases, xsimd implements fallbacks for missing operations on each microarchitecture.

This is where our compile-time plan pays off again.
Fallbacks often require extra computation to emulate missing instructions.
When xsimd knows the inputs at compile time, it can choose simpler fallback paths or precompute transformations, minimizing runtime overhead.

As part of this work, we contributed several fallback improvements to xsimd, specifically for the `swizzle` function.
On AVX2 (widely available on modern x86 CPUs), 256-bit registers are divided into two 128-bit *lanes*.
The CPU's byte shuffle instruction (`_mm256_shuffle_epi8`) cannot move bytes across the lane boundary — it performs two independent 16-byte permutations, one per lane.
Permutation indices must be provided per-lane rather than for the whole register.

When xsimd knows the permutation indices at compile time, it can implement an optimized fallback.
The pseudocode below shows the logic; each line is prefixed with `[C]` (compile time) or `[R]` (runtime).
The actual implementation is in the [xsimd repository](https://github.com/xtensor-stack/xsimd/blob/4d185a6de75ff51fd466064c8d91557ae459105a/include/xsimd/arch/xsimd_avx2.hpp#L1326).

```
// When Magik index is given, the resulting byte is filled with zero
[C] Define Magik = 255
[C] If indices do not require data to change lane:
        [C] Compute per lane indices taking modulo 16
        [R] Permute the input data in each lane
[C] Else if all indices point to low lane data:
        [R] Copy low lane in high lane
        [R] Permute the input data in each lane
[C] Else if all indices point to high lane data:
        [R] Copy high lane in low lane
        [C] Compute per lane indices taking modulo 16
        [R] Permute the input data in each lane
[C] Else:
        [R] Swap both lane in a second register
        [C] Compute self lane indices, replacing cross lane indices with Magik,
            taking self lane indices modulo 16.
        [C] Compute cross lane indices, replacing self lane indices with Magik,
            taking cross lane indices modulo 16.
        [R] Permute input data using self indices
        [R] Permute swapped lane data using cross lane indices
        [R] Merge input and swapped register with bitwise OR
```

This is one of the many zero-cost abstractions that xsimd provides.
From the user's perspective, it's just a single `xsimd::swizzle` call.
If someone contributes a better approach, Apache Arrow benefits from the improvement automatically.

With the plan's nested loops unrolled at compile time, we can now run the microkernel.
However, the microkernel has two constraints:

1. It always extracts a fixed number of values, so the caller must ensure there are enough values remaining.
2. It expects packed values to start on a byte boundary.

This is where `unpack_exact` comes in. It handles two cases:

1. Unpack the remaining values once we've run as many kernel iterations as possible.
2. Unpack values at the beginning of the input array until we reach one that starts on a byte boundary.

When distributing compiled packages for a wide variety of platforms (as with Python and Conda packages), we don't know which SIMD instructions will be available on the client machine.
Rather than restricting ourselves to a minimal baseline, we compile kernels for a set of possible architectures.
At runtime, we select the kernel compiled for the most advanced available microarchitecture — and, as mentioned earlier, for the appropriate packing bit width.
This dispatch has a small upfront cost, but extracting as many values as possible in a single call amortizes it and delivers the best performance.

## Performance Tuning

### Integer size

Despite xsimd's fallbacks, unpacking to certain integer types performs better than others.
Of course, comparing `uint16_t` and `uint64_t` directly isn't fair — the latter writes more zeros.
However, in some cases we can unpack to a different integer type in a local buffer, then copy and cast to the output.

- We can unpack to a *larger* integer (e.g., `uint64_t` when the target is `uint16_t`), then cast and write to the output.
- When the packing width is small, we can unpack to a *smaller* integer (e.g., `uint16_t` when the target is `uint64_t`, if the packing bit size is less than 16).

This tuning depends on the CPU microarchitecture and is driven by benchmarks.

### Batch sizes

When unpacking 1-bit values on a 256-bit register, a single kernel invocation can extract 256 values.
With the algorithm described here, the kernel only runs when the number of values is a multiple of this batch size.
In Apache Parquet, this granularity is often too coarse.
We want the SIMD loop to run even when extracting fewer values, because the packing bit size isn't static — it changes throughout the data.
For example, 512 values might be packed at 3 bits, followed by 200 values at 10 bits, and so on.

We therefore adjust the planning algorithm for small packing bit sizes to reduce the batch size and prevent out-of-bounds reads.

## Generated Assembly

Now that we've seen plan structures, let's examine how they compile to assembly.
The following shows the generated code for unpacking 13-bit values into 32-bit integers on AVX2.
AVX2 uses 256-bit registers named `ymm0`–`ymm15`; in the assembly, `ymmword` indicates a 256-bit memory operand matching this register size.
This is a medium plan (byte spread fits in the output slot), with AVX2 adding a small extra complexity because its shuffle instruction `vpshufb` [can only move bytes within each 128-bit half of the register, not across the boundary](https://www.felixcloutier.com/x86/pshufb).

```asm
        instruction  operands
 1      vpcmpeqd     ymm3, ymm3, ymm3
 2      vmovdqa      ymm8, ymmword [.LC212]
 3      vmovdqa      ymm7, ymmword [.LC213]
 4      vmovdqa      ymm4, ymmword [.LC169]
 5      vmovdqa      ymm6, ymmword [.LC215]
 6      vpsrld       ymm3, ymm3, 0x13
 7      vmovdqa      ymm5, ymmword [.LC216]
 8   ┌► nop          DWORD PTR [rax + rax]
 9   │  vmovdqu      ymm0, ymmword [rcx]
10   │  sub          edi, 0x10
11   │  add          rcx, 0x1a
12   │  add          rsi, 0x40
13   │  vperm2i128   ymm2, ymm0, ymm0, 1
14   │  vpshufb      ymm1, ymm0, ymm8
15   │  vpshufb      ymm0, ymm0, ymm6
16   │  vpshufb      ymm9, ymm2, ymm7
17   │  vpshufb      ymm2, ymm2, ymm5
18   │  vpor         ymm1, ymm1, ymm9
19   │  vpor         ymm0, ymm0, ymm2
20   │  vpsrlvd      ymm1, ymm1, ymm4
21   │  vpsrlvd      ymm0, ymm0, ymm4
22   │  vpand        ymm1, ymm1, ymm3
23   │  vpand        ymm0, ymm0, ymm3
24   │  vmovdqu      ymmword [rsi - 0x40], ymm1
25   │  vmovdqu      ymmword [rsi - 0x20], ymm0
26   │  cmp          edi, 0xf
27   │  jle          <exit>
28   └─ cmp          rax, rcx
```

The structure maps to the plan:

- **Lines 1–7 (before the loop):** Load compile-time constants into registers.
  We can identify each register's role by how it's used later in the loop:
  ymm3 is the mask (used with `vpand` on lines 22–23),
  ymm4 holds shift amounts (used with `vpsrlvd` on lines 20–21),
  and ymm5–ymm8 are swizzle patterns (used with `vpshufb` on lines 14–17).
  These values are derived from the packed bit width, unpacked bit width, and SIMD register size which are all compile-time template parameters.
  Because they're `constexpr`, the compiler embeds them directly as immediate values or loads from fixed addresses.
  If these were runtime parameters, we'd need separate code paths or lookup tables, and the compiler couldn't optimize as aggressively (e.g., dead code elimination, constant folding).
- **Line 8:** Loop entry point. The `nop` is a common compiler optimization: it pads the code so the loop starts at a 16-byte aligned address, improving instruction fetch performance.
- **Lines 9–13:** Load 32 bytes of packed input into ymm0 (`vmovdqu ymm0, ymmword [rcx]`, where brackets mean "memory at address rcx", like `*ptr` in C), update the batch counter and pointers (`sub`, `add`), and copy ymm0's upper 128-bit half into ymm2's *both* halves (`vperm2i128`).
  Why this duplication? AVX2 has a well-known limitation: `vpshufb` can only shuffle bytes *within* each 128-bit half of a register, not across the boundary between them.
  To work around this, we duplicate the upper half so it's accessible in both halves of ymm2.
  (This is an AVX2-specific workaround; AVX-512's `vpermb` can shuffle across the full register.)
- **Lines 14–17:** Apply swizzles (`vpshufb` ×4) to route input bytes to their output slots.
  Why four shuffles? We need to produce two output registers (16 values total), and each output register may need bytes from both 128-bit halves of the input.
  So we shuffle ymm0 twice (lines 14–15) and ymm2 twice (lines 16–17), each with a different precomputed pattern (ymm5–ymm8).
- **Lines 18–19:** Combine the shuffled results with OR (`vpor` ×2).
  Each shuffle could only grab bytes from one 128-bit half.
  Now we OR the partial results together: ymm1 gets bytes that came from the lower half OR the upper half, producing complete output values.
  This is conceptually similar to the large plan's combining step, but here it's working around AVX2's lane restriction rather than byte spread exceeding slot size.
- **Lines 20–21:** Apply per-slot shifts (`vpsrlvd` ×2). The `vpsrlvd` instruction shifts each 32-bit slot by a *different* amount (unlike `vpsrld` which shifts all slots equally). We have two shift instructions because we now have two output registers. The shift amounts come from ymm4, which holds the precomputed plan data.
- **Lines 22–23:** Apply mask (`vpand` ×2) to zero out the upper bits and keep only the 13 packed bits in each slot. Two masks for two output registers.
- **Lines 24–25:** Store the 16 unpacked values (two 256-bit registers × 8 slots each) to the output buffer (`vmovdqu` ×2). The output pointer `rsi` was incremented earlier (line 12).
- **Lines 26–28:** Loop condition: check if at least 16 values remain (`cmp edi, 0xf`) and if the input pointer is still within bounds (`cmp rax, rcx`). If both conditions hold, jump back to line 8.

Notice there are no inner loops in the assembly.
The plan's nested structure (multiple reads, swizzles per read, shifts per swizzle) compiles down to straight-line code — all the iteration is resolved at compile time via template instantiation and fold expressions.
The only loop visible here (lines 8–28) is the outer runtime loop that calls the microkernel repeatedly to process batches of 16 values.
The microkernel itself was inlined into `unpack_width()`, eliminating function call overhead.

For comparison, here is the **old implementation** produced by a Python code generator:

```asm
        instruction  operands
 1      push         rbp
 2      mov          rbp, rsp
 3      and          rsp, 0xfffffffffffffff0
 4      sub          rsp, 0x20
 5      mov          r8d, dword [rdi + 4]
 6      mov          ecx, dword [rdi + 8]
 7      mov          rax, qword fs:[0x28]
 8      mov          qword [var_18h], rax
 9      mov          eax, dword [rdi]
10      mov          edx, r8d
11      vmovd        xmm4, ecx
12      shld         edx, eax, 6
13      vmovd        xmm5, eax
14      vmovd        xmm2, edx
15      mov          edx, ecx
16      shld         edx, r8d, 0xc
17      vpinsrd      xmm2, xmm2, r8d, 1
18      vmovd        xmm1, edx
19      mov          edx, dword [rdi + 0xc]
20      vpinsrd      xmm1, xmm1, ecx, 1
21      shld         edx, ecx, 5
22      vpinsrd      xmm0, xmm4, edx, 1
23      vpunpcklqdq  xmm1, xmm1, xmm0
24      vpinsrd      xmm0, xmm5, eax, 1
25      vpunpcklqdq  xmm0, xmm0, xmm2
26      vinserti128  ymm0, ymm0, xmm1, 1
27      vpsrlvd      ymm1, ymm1, ymmword [.LC57]
28      vpcmpeqd     ymm1, ymm1, ymm1
29      vpsrld       ymm1, ymm1, 0x13
30      vpand        ymm0, ymm0, ymm1
31      vmovdqu      ymmword [rsi], ymm0
32      mov          ecx, dword [rdi + 0x10]
33      mov          eax, dword [rdi + 0xc]
34      mov          r9d, dword [rdi + 0x14]
35      mov          r10d, dword [rdi + 0x18]
36      mov          edx, ecx
37      vmovd        xmm6, ecx
38      vmovd        xmm7, eax
39      shld         edx, eax, 0xb
40      vpinsrd      xmm3, xmm6, ecx, 1
41      mov          r8d, edx
42      mov          edx, r9d
43      shld         edx, ecx, 4
44      vmovd        xmm2, edx
45      mov          edx, r10d
46      shld         edx, r9d, 0xa
47      vpinsrd      xmm2, xmm2, r9d, 1
48      vmovd        xmm0, edx
49      vpinsrd      xmm0, xmm0, r10d, 1
50      vpunpcklqdq  xmm2, xmm2, xmm0
51      vpinsrd      xmm0, xmm7, r8d, 1
52      vpunpcklqdq  xmm0, xmm0, xmm3
53      vinserti128  ymm0, ymm0, xmm2, 1
54      vpsrlvd      ymm0, ymm0, ymmword [.LC59]
55      vpand        ymm0, ymm0, ymm1
56      vmovdqu      ymmword [rsi + 0x20], ymm0
```

This code mixes scalar and SIMD instructions: it loads values one at a time (`mov`), shifts them with scalar operations (`shld`), and manually inserts them into SIMD registers (`vpinsrd`, `vmovd`).
There is no loop — it processes exactly 16 values and returns, requiring the caller to loop externally.
At 56 instructions versus 28 for the new implementation, and without inlining, the difference is substantial.

## Results

Benchmarks were conducted on two machines — an Apple MacBook Pro M3 and a Linux x86-64 Scaleway bare metal instance with AVX2 — across several implementations ranging from scalar baselines to SIMD-optimized variants (`Old`, `New`, `LittleIntPacker`, `Dynamic`), compiled for multiple microarchitectures (`Neon`, `SSE4.2`, `AVX2`).
Results can be explored by machine, function, unpacked integer type, and packed bit width on the [full results page](results.ipynb).

In all benchmarks, cpu times are an affine function of the number of integer to unpack.
We fit linear regression models to these lines in order to compute the asymptotic unpacking speed.
For a small number of integer to unpack, the performance may vary.
Below is a summary of these speed calculations for the main settings.

In all benchmarks, CPU time is an affine function of the number of integers to unpack, so we fit linear regression models to compute the asymptotic unpacking speed (performance may vary for small counts).
The plots below summarizes these speeds across the main settings, comparing four implementations: **ScalarBatch** (SIMD-free batch scalar baseline), **Old** (the prior Apache Arrow SIMD implementation), **New** (the new SIMD implementation introduced in this work), and **LittleIntPacker** (Daniel Lemire's [horizontal packing library](https://github.com/fast-pack/LittleIntPacker), x86-64/SSE only).

#### Speed on Linux x86-64 with AVX2 (higher is faster)
On the plot below showing unpacking speed on the vertical axis and packed bit size on the horizontal axis, we can see the performance curve of various implementations.

The new algorithm described in this post (in green) widely outperforms others, especially on AVX2.

The `LittleIntPacker` library performs well on `uint32` (sometimes better than our new `SSE4.2` implementation) for which it was designed.
For other integer type, the fallback with a `static_cast` loop (also previously used in Arrow C++) is very penalizing.

In many case, even the manually unrolled `ScalarBatch` function is the worst contender.

![Speed results for Linux x86-64](static/speed-linux-64.svg)

#### Speed on MacBook Pro M3 (higher is faster)
Similarily the plot below show performance curves on a MacBook Pro M3.

The new version surpases the old and scalar one for `uint8` and `uint16`.
Surprisingly, the picture is not as clear on `uint32` and `uint64` where even the scalar function performs well.
Understanding why this is the case would require further investigation.

![Speed results for MacOS x86-64](static/speed-osx-arm64.svg)